In [1]:
using Random
using LinearAlgebra
using ProgressMeter
using JLD2
using MosekTools
using Ket
using JuMP

using ppt2

In [2]:
# Load precomputed forms
files_4x4 = [
    "../pncp_forms_4x4.jld2",
]

files_3x3 = [
    "../pncp_forms_3x3.jld2",
]

forms_3x3 = vcat([jldopen(file) do file
    vcat([file[k] for k in keys(file)]...)
end for file in files_3x3]...)

forms_4x4 = vcat([jldopen(file) do file
    vcat([file[k] for k in keys(file)]...)
end for file in files_4x4]...)

forms_4x4[1]

16×16 Matrix{Float64}:
  0.250562   -0.185018    0.595791   -0.0421582  …   0.0121876    -0.0844437
 -0.185018    0.242161   -0.725767    0.0369451      0.11218       0.0838693
  0.595791   -0.725767    2.45581    -0.100628       0.208687     -0.4419
 -0.0421582   0.0369451  -0.100628    0.0861272     -0.4419        0.0523178
 -0.260352    0.243012   -0.959363    0.239431      -0.0108145     0.172371
  0.243012   -0.29567     1.00193    -0.242058   …  -0.301265     -0.206355
 -0.959363    1.00193    -3.15661     0.833063      -0.661692      0.647687
  0.239431   -0.242058    0.833063    0.0179045      0.647687     -0.389412
  0.121636   -0.0532369   0.110013   -0.117369      -0.0413898    -0.0591179
 -0.0532369  -0.0175779   0.203736    0.0851487     -0.025026      0.0648484
  0.110013    0.203736   -1.24314    -0.247287   …  -0.250823      0.262551
 -0.117369    0.0851487  -0.247287   -0.0722558      0.262551      0.111234
 -0.124737    0.0615256   0.0121876  -0.0844437     -0.0007743

In [ ]:
function test_ppt2_pncp(n::Int, m::Int, forms; n_trials::Int=10000, tol::Float64=1e-6)
    @showprogress for i in 1:n_trials
        ppt = rationalize.(ppt2.rand_ppt(n, m), tol=1)
        cpp = ampliation(ppt, ppt, n, m)
        r, idx = findmin(tr.(forms .* Ref(cpp)))
        if r < -tol
            println("Witness $idx detects entanglement: $r")
            return ppt, forms[idx]
        end
    end
    return nothing, nothing
end

function test_ppt2_dps(n::Int, m::Int; n_trials::Int=10000, tol::Float64=1e-6)
    @showprogress for i in 1:n_trials
        ppt = rationalize.(ppt2.rand_ppt(n, m), tol=1)
        cpp = Hermitian(ampliation(ppt, ppt, n, m))
        r, w = entanglement_robustness(cpp, [n, m], 2; solver=Mosek.Optimizer)
        if r > tol
            println("DPS detects entanglement: $r")
            return ppt, w
        end
    end
    return nothing, nothing
end

test_ppt2_dps (generic function with 1 method)

In [4]:
ppt3_pncp, wit3_pncp = test_ppt2_pncp(3, 3, forms_3x3; n_trials=1000)

Progress: 100%|█████████████████████████████████████████| Time: 0:00:30


(nothing, nothing)

In [5]:
ppt3_dps, wit3_dps = test_ppt2_dps(3, 3; n_trials=1000)

Progress:  12%|████▊                                    |  ETA: 0:02:02┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:  37%|███████████████▎                         |  ETA: 0:01:00┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:  66%|██████████████████████████▉              |  ETA: 0:00:29┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:  79%|████████████████████████████████▍        |  ETA: 0:00:18┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:  87%|███████████████████████████████████▌     |  ETA: 0:00:12┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress: 100%|█████████████████████████████████████████| Time: 0:01:29


(nothing, nothing)

In [6]:
psd4_pncp, wit4_pncp = test_ppt2_pncp(4, 4, forms_4x4; n_trials=1000000)

Progress:  53%|█████████████████████▋                   |  ETA: 0:06:19Excessive output truncated after 524295 bytes.

(nothing, nothing)

In [7]:
ppt4_dps, wit4_dps = test_ppt2_dps(4, 4; n_trials=100)

Progress:   3%|█▎                                       |  ETA: 0:05:06┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress: 100%|█████████████████████████████████████████| Time: 0:05:35


(nothing, nothing)